In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# ---------------- DERİN ÖĞRENME: DENOISING (GÜRÜLTÜ GİDERİCİ) AUTOENCODER ----------------
# Partnerin CNN kullanıyorsa, senin DL mantığın: Kasıtlı Gürültü ve Yeniden İnşa (Denoising)

!pip install tensorflow scikit-learn -q

import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_recall_curve, auc, f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
import warnings
warnings.filterwarnings('ignore')

tf.get_logger().setLevel('ERROR')

print("1. Veri yükleniyor ve özellikler hesaplanıyor...")
file_name = "/content/drive/MyDrive/financial_anomaly_benchmark_data (1).csv"
df = pd.read_csv(file_name)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.ffill().fillna(0)

# Özellik Mühendisliği (Z-Skorları)
rolling_vol = df['Volume_Change'].abs().rolling(window=20)
df['Volume_Z_Score'] = (df['Volume_Change'].abs() - rolling_vol.mean()) / (rolling_vol.std() + 1e-8)

rolling_vola = df['Volatility_HighLow'].rolling(window=20)
df['Vola_Z_Score'] = (df['Volatility_HighLow'] - rolling_vola.mean()) / (rolling_vola.std() + 1e-8)

delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).ewm(alpha=1/14, adjust=False).mean()
loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/14, adjust=False).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

std = df['Close'].rolling(window=20).std()
ma = df['Close'].rolling(window=20).mean()
df['BB_Width'] = (std * 4) / ma
df = df.fillna(0)

# Hedef Etiketler
df['Shock_Score'] = df['Volatility_HighLow'] * df['Volume_Change'].abs()
threshold = df['Shock_Score'].quantile(0.99)
df['y_true'] = (df['Shock_Score'] >= threshold).astype(int)

# Gecikme (Lag) yok, veriler eşzamanlı
features = ['Returns', 'Volume_Z_Score', 'Vola_Z_Score', 'RSI', 'BB_Width']
X = df[features].fillna(0)
y = df['y_true']

train_window_size = 2880
test_window_size = 96
test_indices = np.arange(train_window_size, len(df), test_window_size)

y_test_real = []
y_test_scores = []
y_test_preds = []

print(f"2. Denoising (Gürültü Giderici) Autoencoder Mimarisi Kuruluyor...")

input_dim = len(features)
input_layer = Input(shape=(input_dim,))

# 💡 SİHİRLİ DOKUNUŞ: DENOISING MANTIĞI
# Ağa giren verinin %20'sini rastgele karartıyoruz. Ağ, eksik veriden bütünü görmeyi öğreniyor.
noisy_input = Dropout(0.2)(input_layer)

# Encoder: Hafif genişletip sonra daraltıyoruz ki ilişkileri daha iyi öğrensin
encoded = Dense(6, activation='tanh')(noisy_input)
encoded = Dense(3, activation='tanh')(encoded)

# Decoder: Orijinal (gürültüsüz) veriyi inşa etmeye çalışıyor
decoded = Dense(6, activation='tanh')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

# Modeli derliyoruz
dae_model = Model(inputs=input_layer, outputs=decoded)
dae_model.compile(optimizer='adam', loss='mse')

print(f"3. Denoising Eğitimi Başlatılıyor ({len(test_indices)} döngü)...")
start_time = time.time()

for start_test_idx in tqdm(test_indices, desc="Denoising AE"):

    start_train_idx = start_test_idx - train_window_size
    end_test_idx = min(start_test_idx + test_window_size, len(df))

    X_train_window = X.iloc[start_train_idx:start_test_idx].copy()
    X_test_window = X.iloc[start_test_idx:end_test_idx].copy()

    y_train_window = y.iloc[start_train_idx:start_test_idx].copy()
    y_test_window = y.iloc[start_test_idx:end_test_idx].copy()

    if y_train_window.sum() == 0:
        y_test_real.extend(y_test_window.values)
        y_test_scores.extend(np.zeros(len(y_test_window)))
        y_test_preds.extend(np.zeros(len(y_test_window)))
        continue

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_window)
    X_test_scaled = scaler.transform(X_test_window)

    # Model girdisi olarak KENDİSİNİ veriyoruz, Dropout katmanı içeride gürültüyü yaratıyor
    dae_model.fit(X_train_scaled, X_train_scaled,
                  epochs=2,
                  batch_size=256,
                  shuffle=False,
                  verbose=0)

    train_reconstructed = dae_model.predict(X_train_scaled, verbose=0, batch_size=256)
    train_mse = np.mean(np.power(X_train_scaled - train_reconstructed, 2), axis=1)

    test_reconstructed = dae_model.predict(X_test_scaled, verbose=0, batch_size=256)
    test_mse = np.mean(np.power(X_test_scaled - test_reconstructed, 2), axis=1)

    precisions, recalls, thresholds = precision_recall_curve(y_train_window.values, train_mse)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_threshold_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else np.percentile(train_mse, 99)

    test_preds = (test_mse >= best_threshold).astype(int)

    y_test_real.extend(y_test_window.values)
    y_test_scores.extend(test_mse)
    y_test_preds.extend(test_preds)

end_time = time.time()
inference_time_ms = ((end_time - start_time) / len(test_indices)) * 1000

print("\n4. Başarı Metrikleri Hesaplanıyor...")
roc_auc = roc_auc_score(y_test_real, y_test_scores)
precision, recall, _ = precision_recall_curve(y_test_real, y_test_scores)
final_pr_auc = auc(recall, precision)
final_f1 = f1_score(y_test_real, y_test_preds)

print("\n" + "═"*65)
print(" 🚀 DENOISING (GÜRÜLTÜ GİDERİCİ) AUTOENCODER SONUÇLARI 🚀")
print("═"*65)
print(f"ROC-AUC Skoru                : {roc_auc:.4f}")
print(f"PR-AUC Skoru                 : {final_pr_auc:.4f}")
print(f"F1 Skoru                     : {final_f1:.4f}")
print(f"Ort. Çıkarım Süresi (Pencere): {inference_time_ms:.2f} milisaniye")
print("═"*65)

results_df = pd.DataFrame({
    'Model': ['Denoising Autoencoder (DL Logic)'],
    'ROC_AUC': [roc_auc],
    'PR_AUC': [final_pr_auc],
    'F1_Score': [final_f1],
    'Inference_Time_ms': [inference_time_ms]
})
results_df.to_csv('/content/drive/MyDrive/benchmark_results_12.csv', mode='a', header=False, index=False)
print("✅ Denoising AE sonuçları CSV'ye eklendi!")

1. Veri yükleniyor ve özellikler hesaplanıyor...
2. Denoising (Gürültü Giderici) Autoencoder Mimarisi Kuruluyor...
3. Denoising Eğitimi Başlatılıyor (1439 döngü)...


Denoising AE: 100%|██████████| 1439/1439 [09:31<00:00,  2.52it/s]



4. Başarı Metrikleri Hesaplanıyor...

═════════════════════════════════════════════════════════════════
 🚀 DENOISING (GÜRÜLTÜ GİDERİCİ) AUTOENCODER SONUÇLARI 🚀
═════════════════════════════════════════════════════════════════
ROC-AUC Skoru                : 0.8844
PR-AUC Skoru                 : 0.1149
F1 Skoru                     : 0.1488
Ort. Çıkarım Süresi (Pencere): 397.42 milisaniye
═════════════════════════════════════════════════════════════════
✅ Denoising AE sonuçları CSV'ye eklendi!
